# 10 - Explanations: SHAP on the trust score, and explanation quality

**GPU optional. Run all; idempotent.**

Attributions come from XGBoost's built-in TreeSHAP (`pred_contribs`), which is
*exact* for tree ensembles and handles the categorical splits natively - the
external `shap` package's TreeExplainer does not reliably support categorical
XGBoost models. Every attribution is saved per run so figures regenerate.

Most XAI-in-security papers stop at a beeswarm plot. This notebook measures
whether the explanations are any good, on all three seeds:

* **Fidelity** - masking the top-k attributed features (to *missing*, which
  the trees route natively) should move the prediction far more than masking k
  random features. Reported as a ratio.
* **Stability** - near-identical inputs should receive near-identical
  attributions (cosine similarity over nearest-neighbour pairs).
* **Sparsity** - how many features carry 90% of the attribution mass. An
  explanation an analyst can act on is a short one.
* **Agreement** - Spearman rank correlation between SHAP importance and
  XGBoost gain importance, a sanity check that the two views of the model
  agree.

Then **per-regime attribution**: which features drive the score for
no-certificate domains versus certificate-holders. That table is the
explanation of *why* the score is adaptive.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard xgboost scipy

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb, json
from pathlib import Path
from scipy.stats import spearmanr
from src.evaluate import splits
from src.utils import manifest as mf

PRIMARY = 'fusion_c_fused'
SPLITS, SEEDS = ['family_disjoint_v1','random_v1'], [42,43,44]
MODEL_DIR = Path(P['artifacts']['models']); SHAP_DIR = Path(P['artifacts']['shap_values'])
TAB = Path(P['results']['tables']); SHAP_DIR.mkdir(parents=True, exist_ok=True)

X_all = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
probe = set(pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")['domain'])
B = X_all[X_all['domain'].isin(probe)].copy().reset_index(drop=True)

man = mf.load_manifest(P['manifest'])
row = man[man['run_id'] == f'{PRIMARY}_family_disjoint_v1_s42'].iloc[0]
NUM, CATS = list(row['config.num']) if 'config.num' in row else None, None

In [ ]:
# Recover the exact feature lists the fused model was trained with, from
# the model file itself (feature names are stored in the booster).
def load_model(split_name, seed):
    m = xgb.XGBClassifier(); m.load_model(str(MODEL_DIR/f'{PRIMARY}_{split_name}_s{seed}.json'))
    return m
bst = load_model('family_disjoint_v1', 42).get_booster()
FEATS = bst.feature_names
FTYPES = dict(zip(FEATS, bst.feature_types))
CATS = [f for f, t in FTYPES.items() if t == 'c']
NUM  = [f for f in FEATS if f not in CATS]
print(len(FEATS), 'features |', len(CATS), 'categorical:', CATS)

def train_categories(df): return {c: pd.Index(df[c].dropna().unique()) for c in CATS}
def matrix(df, categories):
    Xm = df[NUM].apply(pd.to_numeric, errors='coerce').astype(np.float32)
    for c in CATS:
        known = df[c].where(df[c].isin(categories[c]))
        Xm[c] = pd.Categorical(known, categories=categories[c])
    return Xm[FEATS]

def shap_values(model, Xm):
    """Exact TreeSHAP from XGBoost. Returns (n, n_features) contributions and bias."""
    d = xgb.DMatrix(Xm, enable_categorical=True)
    c = model.get_booster().predict(d, pred_contribs=True)
    return c[:, :-1], c[:, -1]

def predict(model, Xm):
    return model.predict_proba(Xm)[:, 1]

## Compute and save attributions (test part, every run)

In [ ]:
cache = {}
for split_name in SPLITS:
    sp = splits.load_split(P['data']['splits'], split_name); d = sp['domains']
    tr = B[B['domain'].isin(d['train'])]; te = B[B['domain'].isin(d['test'])].reset_index(drop=True)
    categories = train_categories(tr)
    Xte = matrix(te, categories)
    for seed in SEEDS:
        out = SHAP_DIR/f'{PRIMARY}_{split_name}_s{seed}.parquet'
        model = load_model(split_name, seed)
        if out.exists():
            S = pd.read_parquet(out)
            sv = S[FEATS].values
        else:
            sv, bias = shap_values(model, Xte)
            S = pd.DataFrame(sv, columns=FEATS)
            S.insert(0, 'domain', te['domain'].values)
            S['_bias'] = bias; S['label'] = te['label'].values
            S['has_certificate'] = te['has_certificate'].astype(bool).values
            S.to_parquet(out, index=False, compression='zstd')
        cache[(split_name, seed)] = (model, te, Xte, sv, categories)
        print(split_name, seed, sv.shape, 'saved' if not out.exists() else 'loaded')

## Global importance (mean |SHAP|), mean over seeds

In [ ]:
imp = {}
for (split_name, seed), (model, te, Xte, sv, _) in cache.items():
    imp.setdefault(split_name, []).append(np.abs(sv).mean(axis=0))
glob = pd.DataFrame({s: np.mean(v, axis=0) for s, v in imp.items()}, index=FEATS)
glob['std_fd'] = np.std(imp['family_disjoint_v1'], axis=0)
glob = glob.sort_values('family_disjoint_v1', ascending=False).round(4)
display(glob.head(20))
glob.to_csv(TAB/'table_shap_global_importance.csv')

## Per-regime attribution - why the score is adaptive

In [ ]:
rows = []
for (split_name, seed), (model, te, Xte, sv, _) in cache.items():
    hc = te['has_certificate'].astype(bool).values
    for reg, mask in [('nocert', ~hc), ('cert', hc)]:
        a = np.abs(sv[mask]).mean(axis=0)
        for f, v in zip(FEATS, a):
            rows.append({'split': split_name, 'seed': seed, 'regime': reg, 'feature': f, 'mean_abs_shap': v})
reg = pd.DataFrame(rows).groupby(['split','regime','feature'])['mean_abs_shap'].mean().unstack('regime')
fd = reg.loc['family_disjoint_v1'].copy()
fd['share_nocert'] = fd['nocert'] / fd['nocert'].sum()
fd['share_cert']   = fd['cert']   / fd['cert'].sum()
print('Top features per regime (family-disjoint, share of attribution mass)')
print('\nNO-CERTIFICATE regime:'); display(fd.sort_values('share_nocert', ascending=False).head(10).round(4))
print('\nCERTIFICATE-HOLDER regime:'); display(fd.sort_values('share_cert', ascending=False).head(10).round(4))
reg.round(5).to_csv(TAB/'table_shap_per_regime.csv')

lex_feats = [f for f in FEATS if f in ('length','core_length','n_labels','shannon_entropy','vowel_ratio',
    'digit_ratio','hyphen_count','max_consec_consonants','bigram_score','trigram_score',
    'unique_char_ratio','is_idn','has_digit','starts_with_digit')]
cert_feats = [f for f in FEATS if f not in lex_feats]
print()
print('attribution mass on LEXICAL features  - nocert: %.3f | cert: %.3f' %
      (fd.loc[lex_feats,'share_nocert'].sum(), fd.loc[lex_feats,'share_cert'].sum()))
print('attribution mass on CERTIFICATE features - nocert: %.3f | cert: %.3f' %
      (fd.loc[cert_feats,'share_nocert'].sum(), fd.loc[cert_feats,'share_cert'].sum()))

## Explanation quality

In [ ]:
rng = np.random.default_rng(42)

def mask_to_missing(Xm, idx_matrix):
    Xk = Xm.copy()
    for r, cols in enumerate(idx_matrix):
        for c in cols:
            col = FEATS[c]
            Xk.iat[r, Xk.columns.get_loc(col)] = np.nan if col not in CATS else pd.NA
    return Xk

def fidelity(model, Xm, sv, k=5, n=3000):
    idx = rng.choice(len(Xm), min(n, len(Xm)), replace=False)
    Xs, ss = Xm.iloc[idx].reset_index(drop=True), sv[idx]
    p0 = predict(model, Xs)
    top = np.argsort(-np.abs(ss), axis=1)[:, :k]
    rnd = np.stack([rng.choice(len(FEATS), k, replace=False) for _ in range(len(Xs))])
    d_top = np.abs(p0 - predict(model, mask_to_missing(Xs, top))).mean()
    d_rnd = np.abs(p0 - predict(model, mask_to_missing(Xs, rnd))).mean()
    return {'k': k, 'delta_topk': d_top, 'delta_random': d_rnd, 'fidelity_ratio': d_top/max(d_rnd,1e-9)}

def stability(Xm, sv, n_pairs=2000):
    Z = Xm[NUM].apply(pd.to_numeric, errors='coerce').astype(float).values
    Z = (Z - np.nanmean(Z, 0)) / (np.nanstd(Z, 0) + 1e-9); Z = np.nan_to_num(Z)
    sub = rng.choice(len(Z), min(20000, len(Z)), replace=False)
    Zs, ss = Z[sub], sv[sub]
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=2).fit(Zs)
    dist, nbr = nn.kneighbors(Zs[:n_pairs])
    a, b = ss[:n_pairs], ss[nbr[:,1]]
    cos = (a*b).sum(1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1) + 1e-12)
    return {'n_pairs': len(cos), 'mean_cosine': float(cos.mean()), 'median_cosine': float(np.median(cos)),
            'mean_nn_distance': float(dist[:,1].mean())}

def sparsity(sv, thr=0.90):
    a = np.abs(sv); srt = -np.sort(-a, axis=1)
    cum = np.cumsum(srt, 1) / np.clip(srt.sum(1, keepdims=True), 1e-12, None)
    need = (cum < thr).sum(1) + 1
    return {'features_for_90pct_mean': float(need.mean()), 'features_for_90pct_median': float(np.median(need))}

def agreement(model, sv):
    gain = model.get_booster().get_score(importance_type='gain')
    g = np.array([gain.get(f, 0.0) for f in FEATS]); s_ = np.abs(sv).mean(0)
    return {'spearman_shap_vs_gain': float(spearmanr(g, s_).correlation)}

rows = []
for (split_name, seed), (model, te, Xte, sv, _) in cache.items():
    r = {'split': split_name, 'seed': seed}
    r.update(fidelity(model, Xte, sv, k=5)); r.update(stability(Xte, sv))
    r.update(sparsity(sv)); r.update(agreement(model, sv))
    rows.append(r); print(split_name, seed, {k: round(v,3) for k,v in r.items() if isinstance(v,float)})
q = pd.DataFrame(rows)
qtab = q.groupby('split')[['fidelity_ratio','delta_topk','delta_random','mean_cosine',
                            'features_for_90pct_mean','spearman_shap_vs_gain']].agg(['mean','std']).round(4)
display(qtab); qtab.to_csv(TAB/'table_explanation_quality.csv')

## Example explanations (for the paper's case-study figure)

One high-scoring domain per regime and one deferred domain, with their top
contributions. Chosen deterministically (highest / mid-band calibrated score)
so the figure regenerates identically.

In [ ]:
from src.evaluate import predictions
PRED_DIR = Path(P['artifacts']['predictions'])
model, te, Xte, sv, _ = cache[('family_disjoint_v1', 42)]
ts = predictions.load('trustscore_family_disjoint_v1_s42', PRED_DIR).set_index('domain')
te2 = te.assign(score=te['domain'].map(ts['calibrated_score']))

def explain(i, k=6):
    contrib = pd.Series(sv[i], index=FEATS)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index[:k])
    return top.round(4).to_dict()

cases = {
 'highest-score, no-certificate': te2[~te2.has_certificate.astype(bool)]['score'].idxmax(),
 'highest-score, certificate-holder': te2[te2.has_certificate.astype(bool)]['score'].idxmax(),
 'deferral-band example': (te2['score'] - 0.6).abs().idxmin(),
}
for name, i in cases.items():
    r = te2.loc[i]
    print(f'\n{name}: {r.domain}  label={int(r.label)}  score={r.score:.3f}')
    for f, v in explain(i).items(): print(f'   {f:24s} {v:+.4f}')

---

**Reading the quality table.** `fidelity_ratio` >> 1 means the top-attributed
features genuinely drive the prediction; `mean_cosine` near 1 means near-
identical domains get near-identical explanations; a small
`features_for_90pct` means an analyst reads a short explanation. The
per-regime table is the mechanistic account of adaptivity: attribution mass
shifts from lexical features in the no-certificate regime to certificate
features in the certificate-holder regime.

**Next:** `11_figures_tables` - every figure and table from saved artifacts.